# ShelfSense Sprint 3: Baseline VLM (zero-shot)

Runs **Qwen2-VL-2B-Instruct** with no training on the keyframes from Sprint 2 and asks five
audit questions (fridge brand, brands inside, counts, empty space, price tags).

Two modes:
- **image mode**: one question per keyframe
- **video mode**: up to 8 keyframes of a clip given together as one video (the course topic)

### How to run on Kaggle
1. On your laptop (PowerShell, project root), zip the frames:
   `tar -a -c -f data\results\shelfsense_frames.zip -C data\frames fridge_01 fridge_02 fridge_03`
   (list every video folder; do not use `Compress-Archive`, its zips break on Kaggle's Linux)
2. Kaggle → **Datasets → New Dataset** → upload `shelfsense_frames.zip`, name it `shelfsense-frames`
3. Kaggle → **Code → New Notebook** → File → Import Notebook → this `.ipynb`
4. Right panel: **Add Input** → your `shelfsense-frames` dataset.
   **Settings**: Accelerator = **GPU T4 x2** (or P100), **Internet = On** (needs a phone-verified account)
5. **Run All** (about 10–20 min). Then download `vlm_results_<version>.jsonl` from the Output tab
   and save it on your laptop in `data/results/`

### Prompt versions
- **v1**: first baseline. Failure cases: invented brands (Pepsi, 7up in a Coca-Cola fridge), every count = 10,
  invented price lists (`1, 2, 3...`), answers stuck in a loop until the token limit.
- **v2**: same model and settings, only the prompts change: answer only about products behind the glass door,
  list each brand once with a limit, ignore signs and menus, give an explicit "nothing found" answer.

Set `PROMPT_VERSION` in the next code cell. Each version writes its own output file, so results never mix.

In [ ]:
!pip install -q qwen-vl-utils

In [ ]:
import json, time
from pathlib import Path

import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
MAX_FRAMES = 8            # keyframes per clip in video mode
MAX_NEW_TOKENS = 256
PROMPT_VERSION = "v2"     # "v1" = first baseline, "v2" = improved prompts (see the intro cell)
OUT = Path(f"/kaggle/working/vlm_results_{PROMPT_VERSION}.jsonl")

# Each video folder from Sprint 2 has a frames_log.csv; find them wherever Kaggle unzipped the dataset
video_dirs = sorted({p.parent for p in Path("/kaggle/input").rglob("frames_log.csv")})
videos = {d.name: sorted(d.glob("*.jpg")) for d in video_dirs}
for name, frames in videos.items():
    print(f"{name}: {len(frames)} keyframes")
assert videos, "No frames found: did you add the shelfsense-frames dataset as input?"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU in Settings")

In [ ]:
# float16 because the T4 / P100 do not support bfloat16 well. 2B model uses about 5 GB of GPU memory.
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="cuda:0"
).eval()
# Cap image size so a 576x1024 frame stays at full detail but larger photos do not run out of memory
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256 * 28 * 28, max_pixels=1280 * 28 * 28)
print(f"Loaded. GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# One question per KPI. Each asks for a small JSON answer so it can be scored automatically.
# v1 is kept unchanged so its results can be reproduced.
PROMPTS_V1 = {
    "fridge_brand": (
        "This is a photo of a beverage fridge in a shop. Which brand is the FRIDGE itself branded with "
        "(logo or name on the fridge body, header or door), not the drinks inside? "
        'Answer only with JSON: {"fridge_brand": "<brand>"} or {"fridge_brand": null} if there is no branding.'
    ),
    "brands": (
        "List every drink brand you can see on the products INSIDE the fridge. "
        "Only include brands you can actually read or clearly recognise. "
        'Answer only with JSON: {"brands": ["<brand>", ...]}'
    ),
    "counts": (
        "Count the bottles and cans of each drink brand visible inside the fridge (front row facings). "
        'Answer only with JSON: {"counts": {"<brand>": <number>, ...}}'
    ),
    "empty_space": (
        "Is there any empty shelf space or a visible gap where products are missing inside the fridge? "
        'Answer only with JSON: {"empty_space": true or false, "where": "<short description>"}'
    ),
    "price_tags": (
        "Are price tags or price labels visible on the fridge shelves? If yes, read the prices. "
        'Answer only with JSON: {"price_tags_visible": true or false, "prices": ["<price>", ...]}'
    ),
}

# v2: each change targets a v1 failure case
PROMPTS_V2 = {
    "fridge_brand": (
        # v1 answered with shop signs ("Kazfarm Kitchen") and owner text ("Transcom Beverages Limited")
        "This is a photo of a beverage fridge in a shop. Look at the outside of the fridge: the header panel "
        "at the top and the door frame. Which drinks brand logo is printed there? "
        "Ignore shop signs, menu boards and posters on the wall, and ignore small text such as 'property of'. "
        'Answer only with JSON: {"fridge_brand": "<brand>"}, or {"fridge_brand": null} if the fridge has no brand logo.'
    ),
    "brands": (
        # v1 listed brands that are usually sold (Pepsi, 7up, Dr Pepper) and repeated names until cut off
        "Look only at the drinks on the shelves behind the glass door. Read the labels on the bottles and cans. "
        "Include a brand only if you can read its name or see its logo on a product in this image. "
        "Do not guess brands that are commonly sold. The brand on the fridge header does not count unless "
        "its drinks are on the shelves. List each brand once, at most 8 brands. "
        'Answer only with JSON: {"brands": ["<brand>", ...]}'
    ),
    "counts": (
        # v1 answered 10 (or 1) for every brand
        "Look only at the drinks on the shelves behind the glass door. For each brand, count the bottles and "
        "cans you can see in the front row of every shelf. Count one by one; give the exact number you count, "
        "not an estimate. List each brand once, at most 8 brands. "
        'Answer only with JSON: {"counts": {"<brand>": <number>, ...}}'
    ),
    "empty_space": (
        # v1 always said "where": "inside the fridge"
        "Look at each shelf behind the glass door. Is any shelf empty, or is there a clear gap in a row "
        "where bottles or cans are missing? "
        'Answer only with JSON: {"empty_space": true, "where": "<which shelf, counted from the top>"} '
        'or {"empty_space": false, "where": ""}'
    ),
    "price_tags": (
        # v1 always said true and invented sequences (1, 2, 3... / TK 10, TK 15...) or copied "<price>"
        "Price tags are small labels showing a price, attached to the front edge of a shelf inside the fridge. "
        "Menu boards, posters and signs on the wall are not price tags. "
        "Are there price tags on the shelf edges of this fridge? "
        'If you see none, answer {"price_tags_visible": false, "prices": []}. '
        'If you see some, answer {"price_tags_visible": true, "prices": [...]} with at most 5 prices, '
        "each copied exactly as printed. Answer only with JSON."
    ),
}

PROMPTS = {"v1": PROMPTS_V1, "v2": PROMPTS_V2}[PROMPT_VERSION]
print(f"Using prompt version {PROMPT_VERSION}, writing to {OUT}")


def ask(content):
    """Send one user message (images/video + text) and return (answer, seconds)."""
    messages = [{"role": "user", "content": content}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt").to(model.device)
    start = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    seconds = time.perf_counter() - start
    answer = processor.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    return answer.strip(), seconds


def save(row):
    with open(OUT, "a", encoding="utf-8") as f:
        f.write(json.dumps(row) + "\n")


# Resume support: skip anything already answered if the notebook is re-run
done = set()
if OUT.exists():
    for line in OUT.read_text(encoding="utf-8").splitlines():
        r = json.loads(line)
        done.add((r["mode"], r["video"], r["frame"], r["prompt_id"]))
print(f"{len(done)} answers already saved")

### Image mode: every keyframe, every question

In [ ]:
for video, frames in videos.items():
    for frame in frames:
        for pid, prompt in PROMPTS.items():
            if ("image", video, frame.name, pid) in done:
                continue
            answer, sec = ask([{"type": "image", "image": f"file://{frame}"}, {"type": "text", "text": prompt}])
            save({"mode": "image", "video": video, "frame": frame.name, "prompt_id": pid,
                  "answer": answer, "seconds": round(sec, 2), "model": MODEL_ID,
                  "prompt_version": PROMPT_VERSION})
            print(f"[image] {video}/{frame.name} {pid} ({sec:.1f}s): {answer[:100]}")

### Video mode: up to 8 evenly spaced keyframes per clip, as one video

In [ ]:
def pick_evenly(frames, k):
    if len(frames) <= k:
        return frames
    step = (len(frames) - 1) / (k - 1)
    return [frames[round(i * step)] for i in range(k)]


for video, frames in videos.items():
    chosen = pick_evenly(frames, MAX_FRAMES)
    if len(chosen) < 2:  # a video needs at least 2 frames
        chosen = chosen * 2
    for pid, prompt in PROMPTS.items():
        if ("video", video, None, pid) in done:
            continue
        video_prompt = prompt.replace("a photo of", "a video of").replace("this image", "this video")
        content = [
            {"type": "video", "video": [f"file://{f}" for f in chosen], "max_pixels": 640 * 360},
            {"type": "text", "text": video_prompt},
        ]
        answer, sec = ask(content)
        save({"mode": "video", "video": video, "frame": None, "n_frames": len(chosen), "prompt_id": pid,
              "answer": answer, "seconds": round(sec, 2), "model": MODEL_ID,
              "prompt_version": PROMPT_VERSION})
        print(f"[video] {video} ({len(chosen)} frames) {pid} ({sec:.1f}s): {answer[:100]}")

### Quick look

In [ ]:
import pandas as pd

df = pd.read_json(OUT, lines=True)
print(f"{len(df)} answers, median {df.seconds.median():.1f}s per answer, "
      f"peak GPU memory {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
pd.set_option("display.max_colwidth", 120)
df[df["mode"] == "video"][["video", "prompt_id", "answer"]]

### Next
Download `vlm_results_<version>.jsonl` (Output tab, right panel) into `data/results/` on your laptop, then run:
```
python -m src.vlm.evaluate
```
It scores every `vlm_results_*.jsonl` in that folder and prints the versions side by side.